# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt

## 4. Verify Or Prepare Expert-Labelled Data

Download L2DTnH's `2_16000_chatlogs_english_only.csv` to `data/l2dtnh/l2dtnh_english.csv`. The preparation script normalizes the messages, removes contradictory normalized labels, and creates match-grouped train/validation/test splits.

In [ ]:
from pathlib import Path
import subprocess

import pandas as pd

raw_path = Path('data/l2dtnh/l2dtnh_english.csv')
prepared_path = Path('data/l2dtnh/l2dtnh_prepared.csv')

if not raw_path.exists():
    raise FileNotFoundError(
        'Upload 2_16000_chatlogs_english_only.csv as data/l2dtnh/l2dtnh_english.csv.'
    )

# Always rebuild so grouped split assignments and the audit match the current code.
subprocess.run(['python', 'prepare_l2dtnh.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(pd.crosstab(df['split'], df['label']))
print('Groups per split:', df.groupby('split')['group_id'].nunique().to_dict())

## 5. Train LSTM

In [ ]:
!python train.py

## 6. Run Baseline

In [ ]:
!python baseline.py

## 7. Save Checkpoint And Report Evidence To Drive

The repository already lives in Drive if `PROJECT_DIR` points there. This cell also copies the selected weights and small result files into an explicit Colab artifacts folder.

In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path(PROJECT_DIR) / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

checkpoint = Path('checkpoints/best_model.pt')
if checkpoint.exists():
    shutil.copy2(checkpoint, artifact_dir / checkpoint.name)

results_dir = Path('results')
for result in results_dir.glob('*'):
    if result.is_file():
        shutil.copy2(result, artifact_dir / result.name)

print(f'Copied checkpoint and result evidence to {artifact_dir}')